# NITS-IQA analysis (English publication adaptation)

This notebook is adapted from a historical experiment notebook for an English-only repository. The original notebook remains in the local archive. Source data are the 405 recorded pairs; no source images or model inference are included. Numeric calculations retain the historical logic, with an equivalent binary balanced-accuracy implementation and explicit repairs of invalid or out-of-order cells.

In [ ]:
from pathlib import Path
import pandas as pd

def locate_nits_csv():
    """Find the English-header NITS-IQA scores relative to the notebook or repository."""
    base = Path.cwd().resolve()
    candidates = [base / "nits_iqa/verified/NITS_IQA_405_verified.csv",
                  base / "../verified/NITS_IQA_405_verified.csv",
                  base / "verified/NITS_IQA_405_verified.csv"]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError("NITS_IQA_405_verified.csv not found; run from repository root or notebook directory")

DATA_CSV = locate_nits_csv()

def fast_balanced_accuracy(y_true, y_pred):
    """Binary balanced accuracy, equivalent to sklearn on these two-class folds."""
    import numpy as np
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    pos = yt == 1
    neg = yt == 0
    if not pos.any() or not neg.any():
        raise ValueError("Both classes are required for this binary benchmark")
    return 0.5 * ((yp[pos] == 1).mean() + (yp[neg] == 0).mean())


In [ ]:
import pandas as pd

file = DATA_CSV
df = pd.read_csv(file)

# Binary acceptability label based on MOS
df["Acceptable"] = (df["MOS"] >= 50).astype(int)

print("Total Image Pairs:", len(df))
print("Acceptable:", (df["Acceptable"] == 1).sum())
print("Not acceptable:", (df["Acceptable"] == 0).sum())

# Correlation of each metric with MOS
metrics = [
    "DINOv2-Base (%)",
    "DINOv2-Large (%)",
    "Dists (%)",
    "SSIM (%)",
    "OpenCLIP (%)"
]

for metric in metrics:
    correlation = df[metric].corr(df["MOS"], method="spearman")
    print(metric, "SROCC:", round(correlation, 4))